# <span style="color:purple"> Convert Data Units Using TEOS-10 Subroutines </span>

## <span style="color:purple"> Import Data for Conversion </span>


In [1]:
import pandas as pd
import gsw

df = pd.read_parquet("../data/proc/01_imported.parquet")

## <span style="color:purple"> Handle Known Sensor Errors/Issues </span>

### <span style="color:purple"> AM01: Pressure Sensor Failure </span>

<span style="color:purple"> For AM01, pressure sensor data were bad for all 3 instruments, and a constant pressure value was selected for each instrument (from inspection of data near the start in January 2002), and this constant value was used to recalculate conductivity, and in the salinity calculation. Pressure column is -999.0 for all instruments. Assumed pressure of AM01 instruments is: </span> \
<span style="color:purple">cat1969 = 441.1 dbar </span> \
<span style="color:purple">cat1970 = 581.5 dbar </span> \
<span style="color:purple">cat1971 = 743.0 dbar </span>

In [2]:
# Override each pressure for each respective cat with the constant values as above
# .loc is a property of pandas DataFrames that allows you to access and modify specific rows and columns based on labels or boolean conditions. 
# In this case, we are using .loc to select rows where the "borehole" column is "AM01" and the "cat" column matches the respective year (1969, 1970, 1971).
# We then assign the specified pressure values to those rows in the "pressure" column.
df.loc[(df["borehole"] == "AM01") & (df["cat"] == "1969"), "pressure"] = 441.1
df.loc[(df["borehole"] == "AM01") & (df["cat"] == "1970"), "pressure"] = 581.5
df.loc[(df["borehole"] == "AM01") & (df["cat"] == "1971"), "pressure"] = 743.0

### <span style="color:purple"> AM05: Pressure Sensor Failure </span>

<span style="color:purple"> For AM05, pressure sensor data for the second and third downloads was bad for cat7327, and a constant pressure value was selected (the value immediately after deployment in December 2009), and this constant value was used to recalculate conductivity, and in the salinity calculation. Pressure column is -999.0 for all instruments. Assumed pressure of cat7327 at AM05 from 05/02/2011 12:00:01 onwards is: </span> \
<span style="color:purple">cat7327 = 580.2 dbar </span> 

In [3]:
# Override each pressure for the specific cat from the specified data onwards with the constant values as above
df.loc[(df["borehole"] == "AM05") & 
(df["cat"] == "7327") &
(df["time"] >= "2011-02-05 12:00:01"),
"pressure"] = 580.2

### <span style="color:purple"> AM06: Pressure Sensor Failure </span>

<span style="color:purple"> For AM05, pressure sensor data for the second and third downloads was bad for cat7325, and a constant pressure value was selected (the value immediately after deployment), and this constant value was used to recalculate conductivity, and in the salinity calculation. Pressure column is -999.0 for all instruments. Assumed pressure of cat7325 at AM06 from 05/02/2011 09:30:01 onwards is: </span> \
<span style="color:purple">cat7325 = 551.3 dbar </span> 

In [4]:
# Override each pressure for the specific cat from the specified data onwards with the constant values as above
df.loc[(df["borehole"] == "AM06") & 
(df["cat"] == "7325") &
(df["time"] >= "2011-02-05 09:30:01"),
"pressure"] = 551.3

## <span style="color:purple"> Conversions </span>

### <span style="color:purple"> Practical Salinity to Absolute Salinity </span>

In [5]:
# Convert practical salinity to absolute salinity
df["absolute_salinity"] = gsw.conversions.SA_from_SP(
    df["practical_salinity"], # Practical salinity (PSS78)
    df["pressure"],           # Pressure (dbar)
    df["longitude"],          # Longitude of mooring
    df["latitude"]            # Latitude of mooring
)

### <span style="color:purple"> In Situ Temperature to Potential Temperature </span>

In [6]:
df["potential_temperature"] = gsw.conversions.pt0_from_t(
    df["absolute_salinity"],  # Absolute salinity (g/kg)
    df["in_situ_temperature"],# In-situ temperature (°C)
    df["pressure"]            # Pressure (dbar)
)

### <span style="color:purple"> In Situ Temperature to Conservative Temperature </span>

In [7]:
df["conservative_temperature"] = gsw.conversions.CT_from_t(
    df["absolute_salinity"],  # Absolute salinity (g/kg)
    df["in_situ_temperature"],# In-situ temperature (°C)
    df["pressure"]            # Pressure (dbar)
)

## <span style="color:purple"> Calculations </span>

### <span style="color:purple"> In Situ Freezing Point Temperature </span>

In [8]:
df["in_situ_freezing_point_temperature"] = gsw.t_freezing(
    df["absolute_salinity"],  # Absolute salinity (g/kg)
    df["pressure"],           # Pressure (dbar)
    1                         # Saturation fraction (0 to 1, typically 0 for sub-ice shelf ocean cavitys, 1 for open ocean)
)

### <span style="color:purple"> Conservative Freezing Point Temperature </span>

In [9]:
df["conservative_freezing_point_temperature"] = gsw.CT_freezing(
    df["absolute_salinity"],  # Absolute salinity (g/kg)
    df["pressure"],           # Pressure (dbar)
    1                         # Saturation fraction (0 to 1, typically 0 for sub-ice shelf ocean cavitys, 1 for open ocean)
)

### <span style="color:purple"> Save Dataframe as Parquet </span>

In [10]:
# Save to parquet file for efficient storage and future use
# Note: parquet is a columnar storage format that is more efficient for large datasets and preserves data types better than CSV
# Note: parquet files can be read back into pandas with pd.read_parquet("data/proc/02_converted.parquet")
# Note: this will overwrite any existing file at this location
df.to_parquet("../data/proc/02_converted.parquet", index=False)